In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from datetime import datetime

# project dir path 설정
project_dir = "/home/youngjins/project/belief_trading"
os.chdir(project_dir)

# lib 폴더 path append
sys.path.append(project_dir + "/lib")

# lib 폴더 내 모든 모듈 import
from lib.model.tft import TFTModelTrainer, TemporalFusionTransformer

In [3]:
class MarketDataset(Dataset):
    """Custom dataset for market behavior prediction."""

    def __init__(self, df, context_length=5, target_cols=None, transform=True):
        """
        Initialize the dataset.

        Args:
            df (pd.DataFrame): DataFrame containing the market data
            context_length (int): Number of past time steps to consider
            target_cols (list): List of column names for the target variables
            transform (bool): Whether to standardize the OHLCV data
        """
        self.df = df
        self.context_length = context_length

        # Separate features into categories
        self.ohlcv_cols = [
            "open",
            "high",
            "low",
            "close",
            "volume",
            "quote_volume",
            "count",
            "taker_buy_volume",
            "taker_buy_quote_volume",
        ]

        # Action distribution columns (l_0 to l_19 for long, s_0 to s_19 for short)
        self.long_cols = [f"l_{i}" for i in range(20)]
        self.short_cols = [f"s_{i}" for i in range(20)]
        self.agent_action_cols = self.long_cols + self.short_cols  # 40 columns

        # For the target, we predict the next step's agent actions
        if target_cols is None:
            self.target_cols = self.agent_action_cols
        else:
            self.target_cols = target_cols

        # Standardize OHLCV data if needed
        self.transform = transform
        if self.transform:
            self.ohlcv_scaler = StandardScaler()
            self.df[self.ohlcv_cols] = self.ohlcv_scaler.fit_transform(
                self.df[self.ohlcv_cols]
            )

        # We need at least context_length+1 rows to make a single sample
        self.valid_indices = list(range(context_length, len(df)))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        """Get a sample from the dataset."""
        idx = self.valid_indices[idx]

        # Extract context window of past data
        context_start = idx - self.context_length
        context_end = idx

        # Extract OHLCV data for the context window
        ohlcv_data = self.df.iloc[context_start:context_end][self.ohlcv_cols].values

        # Extract agent's own action distribution for the context window
        agent_actions = self.df.iloc[context_start:context_end][
            self.agent_action_cols
        ].values

        # For this implementation, we assume other players' actions are also available
        # In a real-world scenario, you might need to compute this differently
        other_actions = self.df.iloc[context_start:context_end][
            self.agent_action_cols
        ].values

        # Target: next step's agent action distribution
        target = self.df.iloc[idx][self.target_cols].values

        # Convert to torch tensors
        ohlcv_tensor = torch.FloatTensor(ohlcv_data)
        agent_actions_tensor = torch.FloatTensor(agent_actions)
        other_actions_tensor = torch.FloatTensor(other_actions)
        target_tensor = torch.FloatTensor(target)

        return {
            "ohlcv": ohlcv_tensor,
            "agent_actions": agent_actions_tensor,
            "other_actions": other_actions_tensor,
            "target": target_tensor,
            "timestamp": self.df.index[idx],  # Keep timestamp for analysis
        }


def prepare_dataloaders(
    df, context_length=5, batch_size=64, test_ratio=0.2, val_ratio=0.1
):
    """
    Prepare train, validation, and test dataloaders.

    Args:
        df (pd.DataFrame): DataFrame containing the market data
        context_length (int): Number of past time steps to consider
        batch_size (int): Batch size for training
        test_ratio (float): Proportion of data to use for testing
        val_ratio (float): Proportion of training data to use for validation

    Returns:
        tuple: (train_loader, val_loader, test_loader)
    """
    # Sort by timestamp if not already sorted
    if not isinstance(df.index, pd.DatetimeIndex):
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"])
            df.set_index("timestamp", inplace=True)
        else:
            # Assume the first column is the timestamp
            first_col = df.columns[0]
            df[first_col] = pd.to_datetime(df[first_col])
            df.set_index(first_col, inplace=True)

    # Chronological split (time series data should not be randomly split)
    n_samples = len(df) - context_length
    test_size = int(n_samples * test_ratio)
    val_size = int((n_samples - test_size) * val_ratio)
    train_size = n_samples - test_size - val_size

    # Create datasets
    train_df = df.iloc[: train_size + context_length]
    val_df = df.iloc[train_size : train_size + val_size + context_length]
    test_df = df.iloc[train_size + val_size :]

    # Create datasets
    train_dataset = MarketDataset(train_df, context_length=context_length)
    val_dataset = MarketDataset(
        val_df, context_length=context_length, transform=False
    )  # Use same scaling as train
    test_dataset = MarketDataset(
        test_df, context_length=context_length, transform=False
    )  # Use same scaling as train

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader


def analyze_beliefs(model, dataloader, device="cpu"):
    """
    Analyze the belief states learned by the model.

    Args:
        model: The trained TFT model
        dataloader: DataLoader for the data to analyze
        device: Device to run the model on

    Returns:
        dict: Dictionary containing belief analysis results
    """
    model.eval()
    belief_states = []
    timestamps = []
    actions = []

    with torch.no_grad():
        for batch in dataloader:
            ohlcv = batch["ohlcv"].to(device)
            agent_actions = batch["agent_actions"].to(device)
            other_actions = batch["other_actions"].to(device)

            # Forward pass to get predictions and attention weights
            _, attention_weights = model(ohlcv, agent_actions, other_actions)

            # Store the results for analysis
            belief_states.append(attention_weights)
            timestamps.extend(batch["timestamp"])

            # Also store the target actions for correlation analysis
            actions.append(batch["target"].cpu().numpy())

    # Combine results for analysis
    combined_belief_states = {
        "ohlcv_weights": torch.cat([bs["ohlcv_weights"].cpu() for bs in belief_states]),
        "agent_actions_weights": torch.cat(
            [bs["agent_actions_weights"].cpu() for bs in belief_states]
        ),
        "other_actions_weights": torch.cat(
            [bs["other_actions_weights"].cpu() for bs in belief_states]
        ),
        "cross_attention": torch.cat(
            [bs["cross_attention"].cpu() for bs in belief_states]
        ),
    }

    combined_actions = np.concatenate(actions)

    return {
        "belief_states": combined_belief_states,
        "timestamps": timestamps,
        "actions": combined_actions,
    }


def visualize_beliefs(belief_analysis, ohlcv_cols, agent_action_cols):
    """
    Visualize the belief states and their correlation with actions.

    Args:
        belief_analysis: Dictionary from analyze_beliefs function
        ohlcv_cols: List of OHLCV column names
        agent_action_cols: List of agent action column names
    """
    belief_states = belief_analysis["belief_states"]
    timestamps = belief_analysis["timestamps"]
    actions = belief_analysis["actions"]

    # Convert timestamps to datetime if they're not already
    if not isinstance(timestamps[0], datetime):
        timestamps = [datetime.strptime(ts, "%Y-%m-%d %H:%M:%S") for ts in timestamps]

    # 1. Plot variable importance over time
    plt.figure(figsize=(15, 10))

    # OHLCV variable importance
    plt.subplot(3, 1, 1)
    ohlcv_weights = belief_states["ohlcv_weights"].mean(dim=0).squeeze().numpy()
    plt.bar(ohlcv_cols, ohlcv_weights)
    plt.title("OHLCV Feature Importance")
    plt.xticks(rotation=45)

    # Agent actions variable importance
    plt.subplot(3, 1, 2)
    agent_weights = belief_states["agent_actions_weights"].mean(dim=0).squeeze().numpy()
    plt.bar(agent_action_cols, agent_weights)
    plt.title("Agent Actions Feature Importance")
    plt.xticks(rotation=90)

    # Cross-attention importance (nested beliefs)
    plt.subplot(3, 1, 3)
    cross_attn = belief_states["cross_attention"].mean(dim=0).numpy()
    plt.imshow(cross_attn, cmap="viridis")
    plt.colorbar()
    plt.title("Cross-Attention Weights (Nested Beliefs)")

    plt.tight_layout()
    plt.savefig("belief_importance.png")
    plt.close()

    # 2. Plot belief evolution over time
    plt.figure(figsize=(15, 8))

    # Track a few important features over time
    top_ohlcv_idx = np.argsort(ohlcv_weights)[-3:]  # Top 3 OHLCV features
    top_agent_idx = np.argsort(agent_weights)[-3:]  # Top 3 agent action features

    # Plot OHLCV attention over time
    plt.subplot(2, 1, 1)
    for idx in top_ohlcv_idx:
        plt.plot(
            timestamps,
            belief_states["ohlcv_weights"][:, idx].numpy(),
            label=ohlcv_cols[idx],
        )
    plt.title("Top OHLCV Feature Attention Over Time")
    plt.legend()
    plt.xticks(rotation=45)

    # Plot agent action attention over time
    plt.subplot(2, 1, 2)
    for idx in top_agent_idx:
        plt.plot(
            timestamps,
            belief_states["agent_actions_weights"][:, idx].numpy(),
            label=agent_action_cols[idx],
        )
    plt.title("Top Agent Action Attention Over Time")
    plt.legend()
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.savefig("belief_evolution.png")
    plt.close()

    # 3. Correlation between beliefs and actions
    plt.figure(figsize=(12, 10))

    # Calculate correlation between variable importance and action magnitude
    action_magnitude = np.sum(actions, axis=1)

    # OHLCV correlation
    ohlcv_corr = np.zeros(len(ohlcv_cols))
    for i in range(len(ohlcv_cols)):
        ohlcv_corr[i] = np.corrcoef(
            belief_states["ohlcv_weights"][:, i].numpy(), action_magnitude
        )[0, 1]

    plt.subplot(2, 1, 1)
    plt.bar(ohlcv_cols, ohlcv_corr)
    plt.title("Correlation between OHLCV Attention and Action Magnitude")
    plt.xticks(rotation=45)

    # Agent action correlation
    agent_corr = np.zeros(len(agent_action_cols))
    for i in range(len(agent_action_cols)):
        agent_corr[i] = np.corrcoef(
            belief_states["agent_actions_weights"][:, i].numpy(), action_magnitude
        )[0, 1]

    plt.subplot(2, 1, 2)
    plt.bar(agent_action_cols, agent_corr)
    plt.title("Correlation between Agent Action Attention and Action Magnitude")
    plt.xticks(rotation=90)

    plt.tight_layout()
    plt.savefig("belief_action_correlation.png")
    plt.close()


def train_tft_model(
    model, train_loader, val_loader, num_epochs=50, patience=10, device="cpu"
):
    """
    Train the TFT model with early stopping.

    Args:
        model: The TFT model
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        num_epochs: Maximum number of epochs to train
        patience: Number of epochs to wait before early stopping
        device: Device to train on

    Returns:
        model: Trained model
        history: Training history
    """

    model = model.to(device)
    trainer = TFTModelTrainer(model)

    best_val_loss = float("inf")
    counter = 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(num_epochs):
        # Training
        model.train()
        train_losses = []

        for batch in train_loader:
            x_ohlcv = batch["ohlcv"].to(device)
            x_agent_actions = batch["agent_actions"].to(device)
            x_other_actions = batch["other_actions"].to(device)
            y_agent_actions = batch["target"].to(device)

            loss = trainer.train_step(
                x_ohlcv, x_agent_actions, x_other_actions, y_agent_actions
            )
            train_losses.append(loss)

        avg_train_loss = sum(train_losses) / len(train_losses)
        history["train_loss"].append(avg_train_loss)

        # Validation
        model.eval()
        val_losses = []

        with torch.no_grad():
            for batch in val_loader:
                x_ohlcv = batch["ohlcv"].to(device)
                x_agent_actions = batch["agent_actions"].to(device)
                x_other_actions = batch["other_actions"].to(device)
                y_agent_actions = batch["target"].to(device)

                val_loss, _ = trainer.evaluate(
                    x_ohlcv, x_agent_actions, x_other_actions, y_agent_actions
                )
                val_losses.append(val_loss)

        avg_val_loss = sum(val_losses) / len(val_losses)
        history["val_loss"].append(avg_val_loss)

        # Print progress
        print(
            f"Epoch {epoch+1}/{num_epochs}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}"
        )

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            # Save the best model
            trainer.save_model("best_tft_model.pth")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load the best model
    trainer.load_model("best_tft_model.pth")
    return model, history

In [ ]:
# Read the data
df = pd.read_csv("market_data.csv")

# Prepare dataloaders
train_loader, val_loader, test_loader = prepare_dataloaders(
    df, context_length=5, batch_size=64
)

model = TemporalFusionTransformer(
    ohlcv_features=9,  # Adjust based on your data
    agent_actions=40,  # 20 long + 20 short bins
    other_actions=40,  # 20 long + 20 short bins
)

# Train model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, history = train_tft_model(model, train_loader, val_loader, device=device)

# Analyze beliefs
belief_analysis = analyze_beliefs(model, test_loader, device)

# Define column names for visualization
ohlcv_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_volume",
    "count",
    "taker_buy_volume",
    "taker_buy_quote_volume",
]
agent_action_cols = [f"l_{i}" for i in range(20)] + [f"s_{i}" for i in range(20)]

# Visualize beliefs
visualize_beliefs(belief_analysis, ohlcv_cols, agent_action_cols)